# GigaChat и GPT через API
Локальная Qwen запускается в `run_local_model_colab.ipynb`. Её журнал подключается в последней ячейке.

In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import csv
import os
import json
import time
from datetime import datetime, timezone
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from openai import OpenAI
from gigachat import GigaChat
from dotenv import load_dotenv

FOLDER = Path.cwd()
load_dotenv(FOLDER / '.env', override=True)
MODELS = {
    'gigachat': {'model': 'GigaChat-3-Ultra', 'base_url': 'https://api.giga.chat/v1',
                 'key_env': 'GIGACHAT_CREDENTIALS'},
    'gpt': {'model': 'gpt-6.1-sol', 'base_url': 'https://api.llm-gate.tech/v1',
            'key_env': 'LLM_GATE_API_KEY',
            'extra': {'reasoning_effort': 'low', 'max_completion_tokens': 4096}},
}
VERSIONS = {p: version(p) for p in ['openai', 'gigachat', 'python-dotenv']}
clients = {}
SYSTEM = 'Решай задачи по машинному обучению на русском. Отвечай кратко, строго в указанном формате.'


**Запрос:** ответ, токены и время; журнал — `logs/api_responses.jsonl`.


In [ ]:
def ask(name, prompt, temperature=0, question_id='', question_type=''):
    config = MODELS[name]
    parameters = {'model': config['model'], 'temperature': temperature,
                  'messages': [{'role': 'system', 'content': SYSTEM}, {'role': 'user', 'content': prompt}]}
    parameters.update(config.get('extra', {}))
    if name == 'gigachat':
        parameters['temperature'] = max(temperature, 0.0001)
    if name == 'gpt':
        parameters.pop('temperature')
    record = {'date': datetime.now(timezone.utc).isoformat(), 'model': name,
              'id': question_id, 'тип': question_type, 'request': parameters, 'libraries': VERSIONS}
    started = time.perf_counter()
    try:
        if name not in clients:
            key = os.getenv(config['key_env']) or getpass(f'Ключ {name}: ')
            if name == 'gigachat':
                clients[name] = GigaChat(credentials=key, base_url=config['base_url'],
                                         scope=os.getenv('GIGACHAT_SCOPE', 'GIGACHAT_API_PERS'),
                                         ca_bundle_file=str(FOLDER / 'certs' / 'russian_trusted_root_ca.pem'),
                                         timeout=180, max_retries=0)
            else:
                clients[name] = OpenAI(api_key=key, base_url=config['base_url'], timeout=180, max_retries=0)
        for attempt in range(3):
            record['attempts'] = attempt + 1
            try:
                response = (clients[name].chat(parameters) if name == 'gigachat'
                            else clients[name].chat.completions.create(**parameters))
                break
            except Exception as error:
                status = getattr(error, 'status_code', None)
                if name != 'gpt' or status not in (429, 500, 502, 503, 504) or attempt == 2:
                    raise
                delay = min(60, max(1, float(error.response.headers.get('retry-after', 60 if status == 429 else 2))))
                print(f'{question_id} {name}: {status}, повтор через {delay:g} с')
                time.sleep(delay)
        answer = response.choices[0].message.content or ''
        tokens = getattr(response.usage, 'total_tokens', None)
        record.update(answer=answer, tokens=tokens,
                      prompt_tokens=getattr(response.usage, 'prompt_tokens', None),
                      completion_tokens=getattr(response.usage, 'completion_tokens', None),
                      finish_reason=getattr(response.choices[0], 'finish_reason', None))
        if record['finish_reason'] == 'length':
            raise ValueError('Ответ оборван лимитом токенов; частичный текст сохранён в logs/api_responses.jsonl.')
    except (Exception, KeyboardInterrupt) as error:
        record['error'] = str(error) or 'Выполнение прервано'
        raise
    finally:
        record.update(model_id=parameters['model'], seconds=round(time.perf_counter() - started, 3))
        with (FOLDER / 'logs' / 'api_responses.jsonl').open('a', encoding='utf-8') as file:
            file.write(json.dumps(record, ensure_ascii=False) + '\n')
    return answer, tokens, record['seconds']


**Один вопрос:** сними комментарии и выбери модель.


In [ ]:
# answer, tokens, seconds = ask('gpt', 'Чем precision отличается от recall?')
# print(answer)


**Датасет:** передаются вопрос и формат ответа, без эталона. Код типа D здесь не запускается.


In [ ]:
with (FOLDER / 'data' / 'questions.csv').open(encoding='utf-8-sig', newline='') as file:
    questions = list(csv.DictReader(file))

FORMATS = {
    'A': 'Расчёт в 1–3 строках, затем ОТВЕТ: и все запрошенные значения. Без Markdown и LaTeX.',
    'B': 'Ответь «верно» или «неверно» и обоснуй одним коротким предложением.',
    'C': 'Назови метод и необходимые шаги; обоснуй выбор. Не более трёх коротких предложений.',
    'D': 'Только компактный исполняемый Python-код с импортами. Выполни все требования задачи. Без Markdown, комментариев и пояснений.',
    'E': 'Только метка класса, указанная в условии. Без пояснений.',
    'F': 'Короткий ответ с обоснованием, до трёх предложений. Если данных недостаточно, прямо укажи это.',
}

def make_prompt(question):
    return FORMATS[question['тип']] + '\n\n' + question['текст_ru']


**Весь датасет:** запусти нужную группу. Полученные ответы пропускаются. CSV хранит одну запись на строку.


In [ ]:
def save_answers(results):
    path = FOLDER / 'all_answers.csv'
    temporary = path.with_suffix('.tmp')
    with temporary.open('w', encoding='utf-8-sig', newline='') as file:
        writer = csv.DictWriter(file, fieldnames=['id', 'тип', 'model', 'вопрос', 'ответ', 'ошибка'],
                                lineterminator='\n')
        writer.writeheader()
        for row in results.values():
            writer.writerow({key: value.replace('\r\n', '\n').replace('\r', '\n').replace('\n', r'\n')
                             for key, value in row.items()})
    temporary.replace(path)


def load_results():
    results = {}
    by_id = {q['id']: q for q in questions}
    records = [json.loads(line) for filename in ['logs/api_responses.jsonl', 'logs/local_responses.jsonl']
               if (log := FOLDER / filename).exists()
               for line in log.read_text(encoding='utf-8').splitlines()]
    for r in records:
        q = by_id.get(r.get('id'))
        if not q or r['model'] not in set(MODELS) | {'local'} or q['текст_ru'] not in r['request']['messages'][-1]['content']:
            continue
        key = (q['id'], r['model'])
        previous = results.get(key, {})
        if previous.get('ответ') and not previous.get('ошибка'):
            continue
        error = r.get('error', '')
        if r.get('finish_reason') == 'length' or not r.get('answer'):
            error = error or 'Нет полного ответа: запрос прерван или достигнут лимит'
        results[key] = {'id': q['id'], 'тип': q['тип'], 'model': r['model'],
                        'вопрос': q['текст_ru'], 'ответ': r.get('answer', '') if not error else '', 'ошибка': error}
    return results


def run_models(model_names):
    results = load_results()
    save_answers(results)
    for question in questions:
        prompt = make_prompt(question)
        for name in model_names:
            key = (question['id'], name)
            previous = results.get(key, {})
            if previous.get('ответ') and not previous.get('ошибка'):
                continue
            answer, error = '', ''
            try:
                answer, _, _ = ask(name, prompt, question_id=question['id'], question_type=question['тип'])
            except Exception as exception:
                error = str(exception)
            results[key] = {'id': question['id'], 'тип': question['тип'], 'model': name,
                            'вопрос': question['текст_ru'], 'ответ': answer, 'ошибка': error}
            save_answers(results)
            print(question['id'], name, 'готово' if not error else error)


**GigaChat — подключение из РФ.**


In [ ]:
run_models(['gigachat'])


## GPT
Запусти с доступом к зарубежному API.

In [ ]:
run_models(['gpt'])


## Объединить ответы с Colab
Сначала скопируй `logs/local_responses.jsonl` в папку `logs/` проекта. Ячейка обновляет общий CSV без запросов к моделям.

In [ ]:
results = load_results()
save_answers(results)
for name in ['gigachat', 'gpt', 'local']:
    print(name, sum(r['model'] == name and bool(r['ответ']) and not r['ошибка'] for r in results.values()), 'готовых ответов')